# Model proposal for continous integration

<a id="table"></a>

## Table of Contents

- [Aim of the project](#aim)

    - [Outlier definition](#outlier)
    - [Historical processing](#historical_scheme)
    - [Continous setup](#continous_scheme)
    - [Test different approaches](#test)

- [How to use the package](#setup)

    - [Set the envrioment using pip or conda](#envrioment)
    - [Computing capacity required](#power)
    - [Data location](#data)

- [Summary of historical processing method](#method)

    1. [Create lookup table](#lookptable)
    2. [Download satellite images](#donwload)
    3. [Create zarr folder for historical analysis](#create_zarr)
    4. [Run historical NDVI processing](#historic)
    5. [Create COGTIFF file](#tiff)

- [Summary of continous setup method](#continous)


- [Case tested](#case-tested)

    - [Case 1: lowland broadleaf](#lowland-broadleaf)
    - [Case 2: highland broadleaf](#highland-broadleaf)
    - [Case 3: lowland evergreen](#lowland-evergreen)
    - [Case 4: highland broadleaf](#highland-evergreen)
    - [Case 5: fire-affected area](#fire)
    - [Case 6: nearby fire-affected area](#non-fire)
    - [Case 7: 2018 drought-affected area](#drought)
    - [Case 8: Vaia storm-affected area](#storm)




<a name = "aim"></a>

[Go back to the table](#table)

## Aim of the project

In this porject, we create a workflow to process the NDVI data from Sentinel-2 at 10m of spatial resoultion at dailiy scale.
The processing includes:

- filtering of cloud and snow cover

- indvidaution of outlier 

- smoothing of the observation

- linear interpoation of gap between observation

the workflow is done by using already avaible data from April 2017 to November 2025 (historical processing) and a setup to work with new ongoing data (continous setup)

<a name = "outlier"></a>


## Outlier definition


**Outlier definition**

We define an observation to be an outleir if two conditions are met:

- The difference between the absolute NDVI value and the corresponded expected value (hereinafter called median) is above a a threshold (0.1), so called delta.

- The difference between the current analysed delta and the two neighbourh delta is above a threshold (0.1), so called delta-delta.

We decide this defintion so that we are able to correctly idenitfy the seasonal trends and extreme events while flagging the single measurment correctly. 

We tried to take into account the lag between observation, but that results in the impossibility too correctly identify outlier especially during witnter, when the lag is larger.

We tried also to to use the median value in combination of the interquarile range (X + a*IQR) but the resulting htreshold was too large during winter and too thin during summer.

**Potential outlier**

In the continous setup, a new ingested data do not have two neighouboring data. If the new data exceed the aforementioned threshold we defined it as potential outlier.

<a name = "historical_scheme"></a>

## Historical processing

The historical processing follows this streamlined steps:

Download satellite images -> removal of cloud covered or invalid data -> outlier detection -> smoothing and gapfilling

Here we illustrate the workflow for the historical processing. 

<figure>
  <img src="./fig/flowchart_historical.png">
</figure>

<a name = "continous_scheme"></a>


## Continous setup

The continous setup is almost identical to the historical processing. The only difference is that we take into account only the last 7 observation compare to the full time series in the historical processing.

We introduce a new mask for data called *potential outlier*.


We created a bash script that automatically launch all the script in sequence. The script can be found [here](workflow_implementation/demo/test_all_pixels/0_1_run_pipeline.sh) and will perform the following action:

- retrieve the last date analysed by the model
- set the current date as the last date to analyse
- exectue the script 1 to search and download the new satellite images
- if no satellite images are detected, it skip all the computation and overwrite the last date to the current date
- if one (or more) satellite images are found, exectue in order all the script from 2 to 6
- after the computation are finished, it overwrites the last date to the current date

We introduce this dynamic windows of last date analysed and current date to have a system that is flexible (can be either run once per day or with some days lag).

The window date is automatically set from the most recent day not processed yet to the current date at which the bash is activated. Is it possible also tp set the date range by modifiying the aforementioned file. However it would be more efficient to use the script for the historical processing.

Here's the flowchart of the continous setup

<figure>
  <img src="./fig/flowchart_continous.png">
</figure>

<a name = "test"></a>


## Test different approaches

We did some test to find the best solution for the model.

### Smoothing algorithm

We tested 3 different smoothing options

- Low-pass filter

- Savitzky-Golay

- LOESS

We decided to keep the LOESS because the lowe pass filter show some unwnanted spiking during the smoothing and the LOESS was more roubust than the Savitzky-Golay smoothing.


### Smoothing window

We decided to keep a smoothing window of 7 values because if offers the best trade-off between accuracy and performance.

By lowering the window at 5 observation, the smoothing appears to be not effective, by increasing to 9 or above there are no improvement on the smoothing performance.

We set the number of iteration of LOESS algorithm to 3 for the same reason stated above.

In th econtinous integration setup, we will have a rolling window of 7 values, form the last fourth (in the middle) backwards all the data will be smoothed and gapfilled. After that observation the data will be linearly interpolated.

This means that each data is written twice: once when is lineary interpolated and once is smooted.

Below there is a simple scheme of the workflow and the smoothing window.




<a id="setup"></a>

[Go back to the table](#table)

## How to use the package

Here we will describe how to setup the envrioment unsing pip or conda, the computing capacity used and where the data are located

<a id="envrioment"></a>

### Setup the envtioment

The envrioment is already avaible in the folder, it is activated at launch everytime.

It possible to setup the envrioment using Conda or Pip. the following command will create the envrioment based on the preferred choice.

Using pip

``` bash
python3 -m venv ndvi
source ndvi/bin/activate
pip install -r requirements.txt
```
Or using conda
```bash
conda env create -f environment.yml --name ndvi
```
### External packages

Since we are using **imageio** and **rasterio** both **gdal** and **proj** are needed.

It is possible to install them via 

``` bash
sudo apt-get install libgdal-dev libproj-dev
```
or **Conda**

``` bash
conda install gdal proj
```

<a id="power"></a>

### Computing capacity required

We tested the computing capacity required by running the historical processing and the continous setup.

The historical processes takes 23 hours using 80 cores with 20Gb each. The resource of each cores can be lowered.

The final historical setup has a size of 398 GB.

For the continous ingestion, a single date of observation takes ....


<a id="data"></a>

### Data location

The data stored in the demo are just a few pixel to test the script. The historical data are deposit in the following path:

``` bash

/mnt/data2/UniBe-swiss-ndvi/historic_data/historical_2026-04-04_18h16_historical_v7b.zar
```

The lookuptable can be found here

``` bash
/mnt/data2/UniBe-swiss-ndvi/input_data/lookup_table_median_ndvi_v7.zarr
```

<a name = "method"></a>

[Go back to the table](#table)

# Summary of historical processing method

Here, I'll described the method proposed to perform the NDVI processing on the full timeserie from April 2017 to December 2025.


<a name = "lookptable"></a>

## Create lookuptable.py

The analysis includes the donwload of satellite images, the pixel-wise outlier detection, smoothing the observed NDVI and linearly interpolate the missing data between observation at dailiy scale.

To do so, we used the model developed by Samantha, generating the 6 parameters required to calculate the lower and upper double logistic functions. We average the values for each DOY and pixel to create a lookuptable used to perform the analysis.

The pipeline to re-create the lookup table can be started [here](https://github.com/geco-bern/swiss-ndvi-processing/commit/0c877e420a8c88d01152670fd011ed3655d0ab15) (now removed).

The lookup table was computed on our machine and the data are transferred at /mnt/data1/UniBe-swiss-ndvi/data/lookup_table_median_ndvi.zarr. The data dimensions are the pixels and doy.

Here is a schematic of the lookuptable

- Data

    - median NDVI

- Coordinate

    - DOY
    - pixel ID 


<a name = "donwload"></a>

## Extract swisstopo dataset.py


To download the satellite images, we use the pystac_client library in [this script](workflow_implementation/MS1_script_for_historical_NDVI/new_historical_processing/1_download_satellite_images.py)
. We cover the entire Switzerland from 2017-04-01 to 2025-11-30. The data were selected based on the forest mask avaible on Swisstopo VHI dataset.

We apply a filter based on 4 bands, which at least one conditions is met  (green == 9999) | (swir_10m == 9999) (terrain_mask == 255) | (cloud_mask == 255) 

After the filtering, the NDVI and NDSI are computed, the NDVI is filtered out when NDVI >= 0.43. The missing data are flagged with a placeholder of -2^15. The values with no data at given timestep (due to the different orbit) are flagged with a value of 2^16 -1. 

Along with the NDVI and NDSI, we retrieve the date of each image, the pixel ID and spatial idx and coordinates, the final output will look like this:

- Data

    - NDVI
    - NDSI

- Coordinate

    - date
    - pixel ID
    - X and Y ID
    - X and Y coordinates in Swiss system



## Historic NDVI.py

The second scriptcan be found [here](workflow_implementation/MS1_script_for_historical_NDVI/new_historical_processing/2_historical_ndvi_test.py), it will perform the NDVI processing oh historical data. The analysis can be split in three parts:

- outlier detection

- anomalies detection and smoothing

- interpolation at dailiy scale


The outlier detection is the first step to filter out the non-observation values and outliers.

The scaled-down observation must be within 0-1 so it is easy to filter out the missing data with -2^15 or no data with 2^16 -1.

After the first filter is applied, we remove the outliers, we defined an outlier according to this defintion:

- The difference between the absolute NDVI value and the corresponded expected value (hereinafter called median) is above a a threshold (0.1), so called delta.
- The difference between the current analysed delta and the two neighbourh delta is above a threshold (0.1), so called delta-delta.

When both conditions are met, the value is flagged as outlier and is removed from the timeserie.

The following passage is to create the delta timeserie used to linearly interpolate the missing data. This array is created by evaluating each delta on a rolling window of 7 values. 

Within the window, we check if the data inside the window are close to the boundaries conditions or have extreme negative NDVI values, if one (or both) conditions are met the delta is added as it is, otherwise the smoothing is performed. 

the smoothing of the deltas is performed on the non-outlier observed NDVI. We use the LOESS alogorithm to perform the smoothing on a rolling window of 7 observation and 3 iterations of the algoritm. 

After the rolling window reach the last-fourth observation (so that is centered) we cannot proceed using this method, hence we append the remaining deltas that will be flagged as "observation yet to smooth" (L1 linearly interpolated product).

After the delta timeserie is created, we linearly interpolate the results by taking into account their position on the timeserie, the interpolated delta are summed to the medians NDVI to obtain the processed NDVI values.

The final timeserie will have from the third to the last fourth observation the smoothing values (L2) and from the day after the alst fourth observation onwards the linarly delta interpolation (L1).

After the processing, we create the mask array for the TIFF generation. The mask will have integer values from 0 to 4 according to this list

- **0**: the data is not an observation and is yet to be smoothed
- **1**: the data is not an observation and is smoothed
- **2**: the data is an observation and is yet to be smoothed
- **3**: the data is an observation and is smoothed
- **4**: the data is an observation and is an outlier

## TIFF generation

The script to generate the tiff file can be found [here](workflow_implementation/MS1_script_for_historical_NDVI/new_historical_processing/4_plot_historic_tiff.py).

We created two tiff files for each date, one containing the NDVI value and the other containing the mask specified above.



<a name = "continous"></a>

[Go back to the table](#table)

## Summary of continous setup method


The whole workflow to process a single image takes X minutes with Y worker and Z memory. It is equal to the historical setup. The key differences are:

- The merging of historical and newly acquired data before processing

- The decision process highlithed above, after the outlier detection if the new data is an observation the computation is triggered

- After the processing is done, the data are written back in the historical NDVI.

The key parameter that is used in the continous setup:

- The ending date, this parameter can be set manually (as long is higher than the starting date). If not the script will automatically retrieve the current date and set the ending date to the previous day, in order to not skip any new data given a possible daly in release.


The starting and ending date dictate the window in which the data will be processed, the ideal window size contains 7 observation. 

It is possible to include even more observation, however when the number of observation is high it is more efficent to use the historical processing.

The starting and ending date, along with the full computatoin can be launched from this [bash script](workflow_implementation/demo/test_all_pixels/0_1_run_pipeline.sh).

Below, we describe the kwy differences between the historical processing and the continous setup.

The first step is the download and filter of NDVI, is done with [this script](workflow_implementation/demo/test_all_pixels/1_extract_swisstopo_dataset.py) and there is no difference with the historical processing.

The second script performs the merging between historical and newly aqcuired data [here](workflow_implementation/demo/test_all_pixels/4_merge_zarr.py). This script will simply append the new data to perform the analysis.

The [third script](workflow_implementation/demo/test_all_pixels/5_analyse_demo_efficient.py) performs the analysis in the same manner as in the historical processing. 

The only difference is the window of observation, which does not include all the observation but only the observation within the analysis range.

After the analysis is done, we merged the new processed data with the historical data.

The [fourth script](workflow_implementation/demo/test_all_pixels/6_create_cogtiff.py) will create the TIFF ending from the last foruth observation.






<a id="case-tested"></a>

[Go back to the table](#table)

## Case tested

The images are created using the data coming from the historical processing [here](workflow_implementation/demo/test_all_pixels/tmp_extract_timeseries_for_report.py)

We test this model on different biomes and known cases. All pixels are collected and arranged according to the following table. 


| Biome                                 | Coordinates (x, y)  |     
|---------------------------------------|---------------------|
| Lowland broadleaf                     | 2694491, 1126023    |
| Highland broadleaf                    | 2692020, 1121443    |
| Lowland evergreen                     | 2761097, 1194613    |
| Highland evergreen                    | 2781537, 1182974    |
| Biscth fire affected area             | 2644029, 1134128    |
| Biscth fire nearby non-affected area  | 2644328, 1134342    |
| Drought-affected area                 | 2690025, 1287413    |
| Vaia storm affected area              | 2689564, 1154411    |


In [10]:
# This is just to embed the web page and the videos
from IPython.display import IFrame, Video

<a id="lowland-broadleaf"></a>

[Go back to the table](#table)

## Case 1: lowland broadleaf

The broadleaf biomes selected are nicely represent and do not present any complication.

In [ ]:
# area location
x, y = 2694491.82, 1126023.20

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/low_broad.png">
</figure>



<a id="highland-broadleaf"></a>

[Go back to the table](#table)

## Case 2: Highland broadleaf

In [12]:
# area location
x, y = 2692020.28, 1121443.47

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/high_broad.png">
</figure>



<a id="lowland-evergreen"></a>

[Go back to the table](#table)

## Case 3: Lowland evergreen

The lowland evergreen biome selected has high variability (as expcted). With the new set of parameters we are able to represent correctly.

In [13]:
# area location
x, y = 2761097.61, 1194613.45

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/low_ever.png">
</figure>



<a id="highland-evergreen"></a>

[Go back to the table](#table)

## Case 4: Highland evergreen

With the new set of parameters the interquantile range is very large during winter. For that reason we observe a sharp drop during that season. However, LOESS smoothing method has the ability to drastically smooth an extreme value (that we would flag if it was outside the IQR).

In [14]:
# area location
x, y = 2781537.00, 1182975.00 

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/high_ever.png">
</figure>



<a id="fire"></a>

[Go back to the table](#table)

## Case 5: Fire-affected area

With the new set of parameters we are able to correctly flag the drastic reduction in NDVI caused by the fire event. After the event we use the absoulte NDVI to smooth the values instead of the deltas because it is not garantee that the vegetation follows the expected behavior after this drastic event.

In [15]:
# area location
x, y = 2644029.37, 1134128.20 

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/fire.png">
</figure>



<a id="non-fire"></a>

[Go back to the table](#table)

## Case 6: Nearby fire-affected area

The nreaby non affected area by the Bistch fire is clearly different and we (rightfully) do not see the fire event.

In [16]:
# area location
x, y = 2644328.07, 1134342.81

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/non_fire.png">
</figure>



<a id="drought"></a>

[Go back to the table](#table)

## Case 7: Drought affected area

We selected an area north of Schaffausen as illustrated in Fig. 3 in https://onlinelibrary.wiley.com/doi/10.1111/gcb.15360

This drought event mildy affected the vegetation but is possible to see a sharper drop at the summer 2018 season. We are able to correctly indentified the drought and non drought pixels.

In [17]:
# area location
x, y = 2690025.48, 1287413.03

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/drought.png">
</figure>



<a id="storm"></a>

[Go back to the table](#table)

## Case 8: Storm Vaia affected area

The effect of Vaia storm is clearly visible and the NDVI is correctly flagged. It is super interesting to see the different NDVI series after the Vaia storm. The model correctly not flagged any of this different response despite using a nearly identical set of parameters.

The sparce vegetation here again influence the NDVI scattering.

In [18]:
# area location
x, y = 2689564.74, 1154411.88

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/prova3/storm.png">
</figure>

